In [ ]:
#  #219ebc 0%, #8ecae6 100%  #bluegree gradient


<!-- Header -->
<div style="background: linear-gradient(135deg, #274b8e 0%, #3b6db0 100%); padding: 20px; border-radius: 15px; color: white; margin: 10px 0;">
  <h1 style="margin: 0 10 20px; color: white;">EMIL Sample Management System</h1>  
</div>

<!-- Overview Section -->
<div style="border: 2px solid #dee2e6; border-radius: 12px; padding: 20px; margin-top: 10px;">
  <h2 style="margin-top: 0; color: #333;">Overview</h2>

  <p style="margin: 10px 0; opacity: 0.9;">A digital workflow for automated sample creation, data upload, and relationship visualization.</p>  
  
  <p style="margin: 5px 0; opacity: 0.9;">Select your project from the dropdown menu and the following functionalities are available:</p>  
  <div style="
    display: flex;
    justify-content: flex-start;
    gap: 20px;
    margin-top: 10px;
    flex-wrap: wrap;
    max-width: 1000px;
  ">
    <div style="flex: 1 1 250px; padding: 15px; background: #f8f9fa; border-left: 4px solid #28a745; border-radius: 8px; min-width: 250px;">
      <h4 style="margin-top: 0;">📋 Sample Management</h4>
      <p>Create and manage samples through an interactive table interface</p>
    </div>
    <div style="flex: 1 1 250px; padding: 15px; background: #f8f9fa; border-left: 4px solid #007bff; border-radius: 8px; min-width: 250px;">
      <h4 style="margin-top: 0;">📤 Data Upload</h4>
      <p>Upload and associate measurement data files with specific samples</p>
    </div>
    <div style="flex: 1 1 250px; padding: 15px; background: #f8f9fa; border-left: 4px solid #dc3545; border-radius: 8px; min-width: 250px;">
      <h4 style="margin-top: 0;">🕸️ Sample Network</h4>
      <p>Visualize sample and measurement relationships and processing history</p>
    </div>
  </div>
</div>


---
<p style="margin: 5px 0; opacity: 0.9;">Select your project.</p> 

In [ ]:
measurement_types = ["General","XRD","XRR","Bragg-Brentano XRD", "GIXRD","FTS-FID","FTS-TCD", "Fischer-Tropsch Synthesis", "Incipient wetness impregnation","ALD", "XAS", "HAXPES", "XRF", "XPS", "Physisorption", "IR", "Annodization", "Microscopy", "Raman", "UVvis", "SEM", "TEM", "Sputtering", "PECVD", "NanoFab-Oxford Recipe", "Estrellas-Oxford Recipe" ]

In [ ]:
import os, sys
from api_calls import *
import ipywidgets as widgets
from ipyaggrid import Grid 
from datetime import datetime
import pandas as pd
import numpy as np
import time 
import json
from datetime import datetime

url_base = "http://nomad04.csn29.bessy.de"
url_base_external = "https://nomad-se-ais.helmholtz-berlin.de"


proxies = {
    "http": "http://proxy.csn29.bessy.de:3128",
    "https": "http://proxy.csn29.bessy.de:3128",
}



url = f"{url_base}/nomad-oasis/api/v1"
token = os.environ['NOMAD_CLIENT_ACCESS_TOKEN']
uploads = get_all_uploads(url, token, number_of_uploads=100)
time.sleep(1)
date_picker = widgets.NaiveDatetimePicker(value=datetime.now(),disabled=False)

out = widgets.Output()
out2 = widgets.Output()
out3 = widgets.Output()
out4 = widgets.Output()
#upload_names = widgets.Dropdown(options=[u.get("upload_name","--no-name--") for u in uploads],description='Upload:')

upload_names = widgets.Dropdown(
    options=[u.get("upload_name", "--no-name--") for u in uploads if u.get("upload_name")],
    description='NOMAD Upload:',
    layout=widgets.Layout(width='600px', height='60px'),  # adjust the heigh, so it doesnt get cut
    style={'description_width': 'initial'}  # lets label take natural width
)

display(upload_names)

---

In [ ]:
def get_entryids(url, token, sample_ids):  # give it a batch id
    # get al entries related to this batch id
    query = {
        'required': {
            'metadata': '*'
        },
        'owner': 'visible',
        'query': {'results.eln.lab_ids:any': sample_ids},
        'pagination': {
            'page_size': 100
        }
    }
    response = requests.post(
        f'{url}/entries/query', headers={'Authorization': f'Bearer {token}'}, json=query, proxies=proxies)
    data = response.json()["data"]
    
    return [d["entry_id"] for d in data]

def get_specific_data_of_sample(url, token, sample_ids):
    # collect the results of the sample, in this case it are all the annealing temperatures
    entry_ids = get_entryids(url, token, sample_ids)
    
    query = {
        'required': {
            'metadata': '*',
            'data':'*'
        },
        'owner': 'visible',
        'query': {'entry_references.target_entry_id:any': entry_ids,
                'section_defs.definition_qualified_name:any': ['nomad.datamodel.metainfo.basesections.v1.Activity', 'nomad.datamodel.metainfo.basesections.Activity']
                 },
        'pagination': {
            'page_size': 1000
        }
    }
    response = requests.post(f'{url}/entries/archive/query',
                             headers={'Authorization': f'Bearer {token}'}, json=query, proxies=proxies)
    linked_data = response.json()["data"]
    try:
        linked_data.sort(key=lambda x: datetime.strptime(x["archive"]["data"].get("datetime",''), "%Y-%m-%dT%H:%M:%S.%f%z"),reverse=True)
    except:
        pass
    
    return linked_data 

def make_link(label,upload_id, entry_id):
    link = f'{url_base_external}/nomad-oasis/gui/user/uploads/upload/id/{upload_id}/entry/id/{entry_id}'
    return f'<a href={link} target="_blank">{label} <br>'
    
def get_author():
    response = requests.get(f'{url}/users/me', headers={'Authorization': f'Bearer {token}'}, proxies=proxies)
    special_char_map = {ord('ä'):'a', ord('ü'):'u', ord('ö'):'o', ord('ß'):'s'}
    author_name_short = response.json()["first_name"][:2]+response.json()["last_name"][:2] 
    return author_name_short.translate(special_char_map)

def process_upload(upload_id, output=out2):
    response = requests.post(f'{url}/uploads/{upload_id}/action/process', headers={'Authorization': f'Bearer {token}'}, proxies=proxies)
    while True:
        time.sleep(1)
        response = requests.get(f'{url}/uploads/{upload_id}', headers={'Authorization': f'Bearer {token}'}, proxies=proxies)
        if not response.json()["data"]["process_running"]:
            break
        with output:
            print('processing')
    
def get_mainfile(sample_id):
    try:
        entry_id = get_entryid(url, token, sample_id)
        print(entry_id)
        if not entry_id:
            return None
        md = get_entry_meta_data(url, token, entry_id)
        return md.get("mainfile")
    except:
        return None

def get_upload_id():
    upload_id = [u.get("upload_id") for u in uploads if upload_names.value == u.get("upload_name")]
    if len(upload_id) != 1:
        with out2:
            out2.clear_output()
            # print('Could not resolve selected Upload') 
    return upload_id[0]

def get_upload_folder():
    upload_id = get_upload_id()
    for upload_folder in os.listdir(".."):
        if upload_id not in upload_folder:
            continue
        return upload_folder

def get_samples_in_upload(upload_id):
    query = {
        'required': {
            'data': '*'
        },
        'owner': 'visible',
        'query': {'upload_id': upload_id, 'entry_type':"EMIL_Sample"},
        'pagination': {
            'page_size': 150
        }
    }
    response = requests.post(
        f'{url}/entries/archive/query', headers={'Authorization': f'Bearer {token}'}, json=query, proxies=proxies)
    data = response.json()["data"]
    if not data:
        return []
    
    data.sort(key=lambda x: x["archive"]["data"].get("lab_id",''))
    return data

def get_specific_entrytype_of_upload(url, token, upload_id, entry_type, with_meta=False):   
    # in comparison to the query above this method requires the exact nomad entry_type (e.g. CE_NOME_Chronoamperometry instead of Chronoamperometry)
    query = {
        'required': {
            'data': '*',
        },
        'owner': 'visible',
        'query': {
            'upload_id': upload_id,
            'entry_type': entry_type
        },
        'pagination': {
            'page_size': 10000
        }
    }
    response = requests.post(f'{url}/entries/archive/query',
                             headers={'Authorization': f'Bearer {token}'}, json=query, proxies=proxies)
    linked_data = response.json()["data"]
    res = []
    for ldata in linked_data:
        res.append(ldata["archive"]["data"])
    return res 
    

## 📋 Sample Management - Create and Edit Samples

In [ ]:
# Enhanced styling for the interface
from IPython.display import display, HTML

# Add custom CSS for better appearance
display(HTML("""
<style>
    /* General styling improvements */
    .jupyter-widgets {
        font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    }
    
    /* Button styling */
    .widget-button {
        background: linear-gradient(135deg, #274b8e 0%, #3b6db0 100%) !important;
        border: none !important;
        border-radius: 8px !important;
        color: white !important;
        font-weight: 600 !important;
        transition: all 0.3s ease !important;
        box-shadow: 0 4px 15px rgba(102, 126, 234, 0.3) !important;
    }
    
    .widget-button:hover {
        transform: translateY(-2px) !important;
        box-shadow: 0 8px 25px rgba(102, 126, 234, 0.4) !important;
    }
    
    /* Dropdown styling */
    .widget-dropdown select {
        border: 2px solid #e1e8ed !important;
        border-radius: 8px !important;
        padding: 8px 12px !important;
        font-size: 14px !important;
    }
    
    /* Output area styling */
    .jupyter-widgets-output-area {
        background: #f8f9fa;
        border-radius: 10px;
        padding: 15px;
        margin: 10px 0;
        border-left: 4px solid #28a745;
    }
    
    /* AG Grid custom styling */
    

</style>
"""))

columns=["id", "parent (nomadID)", "name","material_formulas","substrate_type", "substrate_dimension","active_area [cm**2]",  "description", "Process and Measurement Data"]

out.clear_output()
out2.clear_output()
df = None
grid = None

def create_sample(upload_folder,data, i ):
    archive =  {
            "data":{
                "m_def":"nomad_hzb_emil.schema_packages.emil_lab_package.EMIL_Sample",
                "name": f'{data["name"]}',
                "datetime": date_picker.value.strftime('%Y-%m-%d %H:%M:%S.%f'),
                "substrate":{"substrate_type":data["substrate_type"], 
                             "substrate_dimension":data["substrate_dimension"]},
                "description": data["description"],
                "active_area": float(data["active_area [cm**2]"]) if data["active_area [cm**2]"] else None,
                "components":[
                    {"m_def":"nomad.datamodel.metainfo.basesections.PureSubstanceComponent",
                    "pure_substance":{
                        "m_def":"nomad.datamodel.metainfo.basesections.PureSubstanceSection",
                        "molecular_formula":formula.strip()
                    }} for formula in data["material_formulas"].split(",") 
                ] if data["material_formulas"] else None
               }}
    
    if data["parent (nomadID)"]:
        archive["data"].update({"parent":{"lab_id":data["parent (nomadID)"] }})
    if data["id"]:
        archive["data"].update({"lab_id": data["id"]})


    
    import json
    archive_name = f'{i}_sample.archive.json'
    with open(f"../{upload_folder}/{archive_name}", "w") as f:
        f.write(json.dumps(archive))

def get_parameter(d, path):
    if not d:
        return ""
    p_split = path.split("/")
    if len(p_split)==1:
        return d.get(p_split[0], "")
    if isinstance(d, list):
        return get_parameter(d[int(p_split[0])], "/".join(p_split[1:]))
    return get_parameter(d.get(p_split[0], ""), "/".join(p_split[1:]))

def show_samples():
    global grid,df
    upload_id = get_upload_id()
    samples = get_samples_in_upload(upload_id)  
    sample_ids = [s["archive"]["data"]["lab_id"] for s in samples if "lab_id" in s["archive"]["data"]]
    sample_data = get_specific_data_of_sample(url, token, sample_ids)

    processes = {}
    
    for d in sample_data:
        if "samples" not in d["archive"]["data"] or "lab_id" not in d["archive"]["data"]["samples"][0]:
            continue
        label= d["archive"]["data"]["samples"][0].get("lab_id")
        if label not in processes:
            processes.update({label:[]})
        d["archive"]["data"].setdefault("method", "Measurement")
        processes[label].append(d["archive"]["data"]["method"])
    df = pd.DataFrame(columns=columns)  
    for s in samples:
        archive_data = s["archive"]["data"]
        sample_id = get_parameter(archive_data, "lab_id")
        row = [
          sample_id, 
          get_parameter(archive_data, "parent/lab_id"), 
          get_parameter(archive_data, "name"),
          ",".join([get_parameter(archive_data, f"components/{i}/pure_substance/molecular_formula")
                   for i in range(len(archive_data.get("components",[])))
                   ]),
          get_parameter(archive_data, "substrate/substrate_type"), 
          get_parameter(archive_data, "substrate/substrate_dimension"), 
          get_parameter(archive_data, "active_area"), 
          get_parameter(archive_data, "description"),
          list(set(processes.get(sample_id,'')))
        ]

        df.loc[len(df)] = row
    for i in range(len(df), df["id"].count()+12):
        df.loc[len(df)] = pd.Series(dtype='object')
    
              
    grid_options = {
        'columnDefs' : [{'headerName':c,'field': c} for c in df.columns],
        'defaultColDef': {'editable': True},
        'rowSelection': 'multiple',
        'enableRangeSelection': True,
    }
    grid = Grid(grid_data=df,grid_options=grid_options,sync_on_edit=True,theme='ag-theme-balham',columns_fit='auto',index=False)
    out.clear_output()
    with out:
        display(grid)
                           
                           
def on_create_clicked(b):
    global df, grid
    out2.clear_output()
    upload_id = get_upload_id()
    upload_folder = get_upload_folder()
    with out2:
        display(HTML("""
        <div style="background: #fff3cd; padding: 10px; border-radius: 8px; border-left: 4px solid #ffc107; margin: 10px 0;">
            <strong>🔄 Processing:</strong> Creating entries (this may take some time)...
        </div>
        """))
    if not grid:
        return
    grid_data = grid.grid_data_out['grid'] if 'grid' in grid.grid_data_out else df
    for idx, row in grid_data.iterrows():
        row = row.replace(np.nan,None)
        print(list(row))
        try:
            idx = idx[0]
        except:
            pass
        if not any(row):
            continue
        try:
            mf = create_sample(upload_folder , row, idx)
        except Exception as e:
            with out2:
                display(HTML(f"""
                <div style="background: #f8d7da; padding: 10px; border-radius: 8px; border-left: 4px solid #dc3545; margin: 5px 0;">
                    <strong>❌ Error:</strong> {e}
                </div>
                """))
            
        if not row["id"]:
            process_upload(upload_id)
    process_upload(upload_id)
    with out2:
        display(HTML("""
        <div style="background: #d4edda; padding: 10px; border-radius: 8px; border-left: 4px solid #28a745; margin: 10px 0;">
            <strong>✅ Success:</strong> Entries created successfully!
        </div>
        """))

    time.sleep(0.5)
    show_samples()
    time.sleep(0.1)
    load_samples_in_upload()

    

button_create = widgets.Button(
    description='Create Entries',
    tooltip='Create sample entries from the data grid',
    layout=widgets.Layout(
        width='200px',
        height='45px'
    ),
    style={'button_color': '#28a745'}
)

time.sleep(0.5)
show_samples()

def on_change_uploads(change):
    out2.clear_output()
    show_samples()
    load_samples_in_upload()

    #on_load_button_clicked(change)

button_create.on_click(on_create_clicked)
upload_names.observe(on_change_uploads, names="value")

# Enhanced upload selection display


display(widgets.VBox([
    #widgets.HBox([upload_names]),
    out, 
    button_create, 
    out2
]))





---

<div style="
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Oxygen, Ubuntu, Cantarell, sans-serif;
    display: flex;
    min-height: 6vh;
    margin: 0;
">
    <div style="
        background: white;
        border-radius: 12px;
        box-shadow: 0 4px 20px rgba(0, 0, 0, 0.1);
        padding: 5px;
        max-width: 200px;
        text-align: center;
        border-left: 6px solid #ff6b6b;
    ">
        <div style="
            font-size: 24px;
            color: #ff6b6b;
            margin-bottom: 5px;
        ">⚠️</div>
        <p style="
            color: #666;
            font-size: 12px;
            line-height: 1.;
            margin-bottom: 5px;
        ">
            If you encounter an error while using this application, please restart the Voila script in <a href="https://nomad-se-ais.helmholtz-berlin.de//nomad-oasis/gui/analyze/north" style="color: #007bff; text-decoration: none; font-weight: 600;" onmouseover="this.style.color='#0056b3'; this.style.textDecoration='underline';" onmouseout="this.style.color='#007bff'; this.style.textDecoration='none';">NOMAD (click me)</a>.
        </p>
    </div>
</div>


## 📤 Data Upload - Upload files and link to samples

In [ ]:
%matplotlib ipympl
%load_ext autoreload
%autoreload 2
import ipywidgets as widgets
from ipyvuetify.extra import FileInput
from ipyvuetify.extra.file_input import ClientSideFile

import hashlib
from IPython.display import display, Markdown, HTML
import os
import sys
import pandas as pd
import numpy as np
import requests
import time
sys.path.append(os.path.dirname(os.getcwd()))
from api_calls import get_nomad_ids_of_entry

# Enhanced styling for upload interface
display(HTML("""
<style>
    /* Enhanced upload interface styling */
    .upload-container {
        background: linear-gradient(135deg, #f093fb 0%, #f5576c 100%);
        border-radius: 15px;
        padding: 20px;
        color: white;
        margin: 15px 0;
        box-shadow: 0 10px 30px rgba(240, 147, 251, 0.3);
    }
    
    .file-upload-button {
        background: linear-gradient(135deg, #4facfe 0%, #00f2fe 100%) !important;
        border: none !important;
        border-radius: 10px !important;
        padding: 15px 30px !important;
        color: white !important;
        font-weight: 600 !important;
        font-size: 16px !important;
        cursor: pointer !important;
        transition: all 0.3s ease !important;
        box-shadow: 0 5px 20px rgba(79, 172, 254, 0.4) !important;
    }
    
    .file-upload-button:hover {
        transform: translateY(-3px) !important;
        box-shadow: 0 10px 30px rgba(79, 172, 254, 0.6) !important;
    }
    
    .sample-button {
        background: linear-gradient(135deg, #274b8e 0%, #3b6db0 100%) !important;
        border: none !important;
        border-radius: 8px !important;
        color: white !important;
        margin: 5px !important;
        transition: all 0.3s ease !important;
        box-shadow: 0 4px 15px rgba(102, 126, 234, 0.3) !important;
    }
    
    .sample-button:hover {
        transform: scale(1.05) !important;
        box-shadow: 0 8px 25px rgba(102, 126, 234, 0.5) !important;
    }
    
    .warning-box {
        background: linear-gradient(135deg, #ff9a56 0%, #ffad56 100%);
        border: none;
        border-radius: 10px;
        padding: 15px;
        color: white;
        margin: 10px 0;
        box-shadow: 0 4px 15px rgba(255, 154, 86, 0.3);
    }
    
    .success-box {
        background: linear-gradient(135deg, #4facfe 0%, #00f2fe 100%);
        border: none;
        border-radius: 10px;
        padding: 15px;
        color: white;
        margin: 10px 0;
        box-shadow: 0 4px 15px rgba(79, 172, 254, 0.3);
    }
    
    .file-list {
        background: #f8f9fa;
        border-radius: 10px;
        padding: 15px;
        border-left: 5px solid #007bff;
        margin: 10px 0;
    }
    
    .panel-container {
        border: 2px solid #e9ecef;
        border-radius: 15px;
        background: #ffffff;
        box-shadow: 0 8px 25px rgba(0, 0, 0, 0.1);
        overflow: hidden;
    }
    
    .panel-header {
        background: linear-gradient(135deg, #f8f8f8 0%, #eaeaea 100%);
        color: #808080;
        padding: 15px 20px;
        margin: 0;
        font-weight: 600;
    }
</style>
"""))

upload_and_process = widgets.Button(
    description="Upload and Process", 
    layout=widgets.Layout(width='200px', height='45px'),
    style={'button_color': '#27008a'},
    tooltip="Upload files to samples and process (not reversible)"
)
#274b8e 0%, #3b6db0
unrecognized_files_widget = widgets.HTML(value="",layout=widgets.Layout(width='500px'))

# Create a container for the upload_and_process button
upload_button_container = widgets.Output()

file_input = FileInput(
    multiple=True,
    label='📁 Click to Select Files',
    v_model=[],
    # Use the 'selection' slot to control what is displayed
    v_slots=[{
        'name': 'selection',
        # This template will display nothing, effectively hiding the file list
        'children': ''
        # Alternatively, show just a file count like this:
        # 'children': '{{ len(props.files) }} file(s) selected'
    }]
)

# Hide the file chips using custom CSS
display(HTML("""
<style>
    .v-chip {
        display: none !important;
    }
    .v-file-input__text {
        display: none !important;
    }
</style>
"""))


file_count_display = widgets.HTML(value="<div style='text-align: center; color: #6c757d; font-style: italic;'>📁 No files selected</div>")

def trigger_file_input(b):
    file_input.fire_event('click', {})
    
#upload_button_custom.on_click(trigger_file_input)

file_selector = widgets.SelectMultiple(
    options=[],
    description='Files:',
    disabled=False,
    layout=widgets.Layout(width='450px', height='500px')
)
dropdown_all_files = widgets.Dropdown(
    options=measurement_types,
    value='General',
    description='Default type for measurments',
    layout=widgets.Layout(width='400px', height='50px'),
    style={'description_width': 'initial'}
)

# Global variables
upload_files = []
sample_id_buttons = []
output_areas = {}
sample_files_dict = {}
file_type_dict = {}  # Dictionary to store file type selections for each sample and file
selected_sample_id = None
raw_upload_data = None
uploaded_files_data = {}  # Store file name to file data mapping


def extract_filenames_from_vuetify(file_data_list):
    """Extract filenames from ipyvuetify FileInput data"""
    filenames = []
    for file_data in file_data_list:
        if isinstance(file_data, dict) and 'name' in file_data:
            filenames.append(file_data['name'])
    return filenames

def on_file_input_change(change):
    """Handle file input change for ipyvuetify FileInput"""
    try:
        global raw_upload_data, uploaded_files_data
        with out4:
            out4.clear_output()
            display(HTML(f"""
            <div class="warning-box">
                <strong>Uploading Data</strong>
            </div>
            """))
        
        # Get the file data from the change event
        file_data_list = file_input.get_files()
        
        if not file_data_list:
            return
            
        # Extract filenames
        filenames = extract_filenames_from_vuetify(file_data_list)
        
        # Store file data for later use
        for file_data in file_data_list:
            if isinstance(file_data, dict) and 'name' in file_data:
                uploaded_files_data[file_data['name']] = file_data
                uploaded_files_data[file_data['name']]["file_content"] = file_data["file_obj"].read()
        
        # Prefilter files
        recognized_files = []
        files_with_dots = []
        
        for filename in filenames:
            # Check for dots in filename (excluding the extension)
            base_name = os.path.splitext(filename)[0]
            if '.' in base_name:
                files_with_dots.append(filename)
            
            # Continue with recognition logic
            filename_lower = filename.lower()
            recognized_files.append(filename)
            
        
        # Update file selector
        file_selector.options = sorted(recognized_files )

        file_count = len(filenames)
        if file_count > 0:
            file_count_display.value = f"<div style='text-align: center; color: #28a745; font-weight: 600;'>📁 {file_count} files selected ✅</div>"
        else:
            file_count_display.value = "<div style='text-align: center; color: #6c757d; font-style: italic;'>📁 No files selected</div>"
        
        with out4:
            out4.clear_output()
            display(HTML(f"""
            <div class="success-box">
                <strong>✅ Upload Successful!</strong><br>
                Uploaded {len(filenames)} files. {len(recognized_files)} files were automatically recognized.
            </div>
            """))
            
            # Display files with dots warning
            if files_with_dots:
                dots_html = """
                <div style="background: #fff3cd; border: 1px solid #ffeaa7; border-radius: 10px; padding: 15px; margin: 10px 0;">
                    <h4 style="margin: 0 0 10px 0; color: #856404;">⚠️ Filename Warning</h4>
                    <p style="margin: 0 0 10px 0; color: #856404;">The following files contain periods (.) which may cause issues:</p>
                    <p style="margin: 0 0 10px 0; color: #856404; font-weight: 600;">Recommendation: Use underscores (_) instead of periods in filenames.</p>
                    <div style="max-height: 150px; overflow-y: auto; background: #f8f9fa; border-radius: 5px; padding: 10px;">
                """
                for file in files_with_dots:
                    dots_html += f"<div style='margin: 2px 0; color: #856404;'>• {file}</div>"
                dots_html += "</div></div>"
                display(HTML(dots_html))
                
    except Exception as e:
        with out4:
            out4.clear_output()
            display(HTML(f"""
            <div style="background: #f8d7da; border: 1px solid #f5c6cb; border-radius: 10px; padding: 15px; color: #721c24;">
                <strong>❌ Error processing upload:</strong> {e}
            </div>
            """))

def on_selection_change(change):
    if change['type'] == 'change' and change['name'] == 'value':
        selected = change['new']
        with out4:
            out4.clear_output()
            if selected:
                display(HTML(f"""
                <div class="file-list">
                    <h4 style="color: #007bff; margin: 0 0 10px 0;">📋 Selected Files</h4>
                    <div style="font-family: monospace;">{', '.join(selected)}</div>
                </div>
                """))
            else:
                display(HTML("<div style='color: #6c757d; text-align: center; font-style: italic;'>No files selected</div>"))

def on_remove_button_click(sample_id, sample_select):
    def handle_click(b):
        global sample_files_dict

        # Get selected files to remove
        selected_files = list(sample_select.value)

        if selected_files:
            # Remove selected files from the sample's list
            sample_files_dict[sample_id] = [f for f in sample_files_dict[sample_id] if f not in selected_files]

            # Update the SelectMultiple widget
            sample_select.options = sample_files_dict[sample_id]

            # Add the removed files back to the file_selector
            file_selector.options = list(file_selector.options) + selected_files

            with out4:
                out4.clear_output()
                display(HTML(f"""
                <div class="success-box">
                    <strong>🗑️ Files Removed from {sample_id}</strong><br>
                    Remaining files: {len(sample_files_dict[sample_id])}
                </div>
                """))

    return handle_click

def on_sample_button_click(sample_id, sample_select):
    def handle_click(b):
        global selected_sample_id, sample_files_dict
        selected_sample_id = sample_id
        selected_files = list(file_selector.value)

        if selected_files:
            # Update the dictionary by appending the selected files
            sample_files_dict[sample_id].extend(selected_files)

            # Update the SelectMultiple widget with the current files
            sample_select.options = sample_files_dict[sample_id]

            # Remove selected files from SelectMultiple widget
            file_selector.options = [f for f in file_selector.options if f not in selected_files]

            # Print the updated dictionary for verification
            with out4:
                out4.clear_output()
                display(HTML(f"""
                <div class="success-box">
                    <strong>📎 Files Added to {sample_id}</strong><br>
                    Total files: {len(sample_files_dict[sample_id])}
                </div>
                """))
    return handle_click

def on_sample_button_first_click(sample_id, output_area):
    def handle_first_click(b):
        global sample_files_dict, file_type_dict

        # Create SelectMultiple widget for this sample
        sample_select = widgets.SelectMultiple(
            options=sample_files_dict[sample_id],
            description='',
            disabled=False,
            layout=widgets.Layout(width='230px', height='100px')
        )

        # Create Add button (for subsequent additions)
        add_button = widgets.Button(
            description="➕ Add",
            layout=widgets.Layout(width='80px', height='35px'),
            style={'button_color': '#274b8e'},
            tooltip="Add selected files to this sample"
        )
        
        # Create Remove button
        remove_button = widgets.Button(
            description="🗑️ Remove",
            layout=widgets.Layout(width='80px', height='35px'),
            style={'button_color': '#dc3545'},
            tooltip="Remove selected files from this sample"
        )

        # Create container for file type dropdowns
        file_type_container = widgets.VBox([], layout=widgets.Layout(margin='10', padding='10', height='200', overflow='scroll'))


        # Function to update the file type dropdowns
        def update_file_types():
            # Initialize file_type_dict entries for this sample if they don't exist
            if sample_id not in file_type_dict:
                file_type_dict[sample_id] = {}

            # Create dropdown widgets for each file
            dropdown_widgets = []
            for file_name in sample_files_dict[sample_id]:
                # Determine default type based on filename
                default_type = dropdown_all_files.value  # Default
                file_lower = file_name.lower()
                for mtype in measurement_types:
                    if mtype in file_lower:
                        default_type = mtype

                # Initialize in dictionary if not present
                if file_name not in file_type_dict[sample_id]:
                    file_type_dict[sample_id][file_name] = default_type

                # Create dropdown for this file
                dropdown = widgets.Dropdown(
                    options=measurement_types,
                    value=file_type_dict[sample_id][file_name],
                    description='',
                    layout=widgets.Layout(width='120px', height='50px')
                )

                # Create observer to update dictionary when dropdown changes
                def make_observer(fname):
                    def observer(change):
                        file_type_dict[sample_id][fname] = change['new']
                    return observer

                dropdown.observe(make_observer(file_name), names='value')

                # Create row with filename and dropdown
                truncated_name = file_name[:15] + '...' if len(file_name) > 20 else file_name
                row = widgets.HBox([
                    dropdown,
                    widgets.HTML(f"<div style='width:150px; font-size:0.9em;'>{truncated_name}</div>"),
                ], layout=widgets.Layout(margin='0', padding='0', height='90px'))

                dropdown_widgets.append(row)

            # Update the container with all dropdown widgets
            file_type_container.children = tuple(dropdown_widgets)

        # Set up callbacks
        add_button.on_click(on_sample_button_click(sample_id, sample_select))
        remove_button.on_click(on_remove_button_click(sample_id, sample_select))

        # Observer for when files are added/removed
        def on_options_change(change):
            update_file_types()

        sample_select.observe(on_options_change, names='options')

        # Initial update of file types
        update_file_types()

        # Display the widgets in the output area
        with output_area:
            output_area.clear_output()

            # Create a horizontal layout with the buttons and sample_select on the left
            # and the file type container on the right
            display(widgets.HBox([
                # Left side: buttons and sample_select
                widgets.HBox([
                    widgets.VBox([
                        add_button,
                        remove_button,
                    ], layout=widgets.Layout(margin='0 10px 0 0')),
                    sample_select
                ]),  # Add left margin for spacing
                widgets.VBox([
                    widgets.HTML("<small style='margin-bottom:5px'><b>Recognized Type:</b></small>"),
                    file_type_container
                ], layout=widgets.Layout(margin='10px 10px 10px 15px', height="200px", overflow='scroll'))  # Add left margin for spacing
            ]))

        # Also handle the initial file transfer
        selected_files = list(file_selector.value)
        if selected_files:
            # Update the dictionary by appending the selected files
            sample_files_dict[sample_id].extend(selected_files)

            # Update the SelectMultiple widget
            sample_select.options = sample_files_dict[sample_id]

            # Remove selected files from file_selector
            file_selector.options = [f for f in file_selector.options if f not in selected_files]

            # Update file types after adding files
            update_file_types()

            with out4:
                out4.clear_output()
                display(HTML(f"""
                <div class="success-box">
                    <strong>🎯 Files assigned to {sample_id}</strong><br>
                    {len(sample_files_dict[sample_id])} files ready for upload
                </div>
                """))

    return handle_first_click

def on_upload_file(b):
    global sample_files_dict, uploaded_files_data, file_type_dict
    upload_ids = []
    # Process files for each sample ID in the dictionary
    for sample_id, file_names in sample_files_dict.items():
        if not file_names:  # Skip if no files assigned to this sample
            continue

        # Get upload ID for this sample
        entry_id, upload_id = get_nomad_ids_of_entry(url, token, sample_id)
        time.sleep(0.2)
        upload_ids.append(upload_id)
        upload_folder = get_upload_folder()
        
        # Process each file assigned to this sample ID
        for file_name in file_names:
            # Get file data from our stored mapping
            if file_name not in uploaded_files_data:
                with out4:
                    display(HTML(f"""
                    <div style="background: #f8d7da; border-radius: 8px; padding: 10px; margin: 5px 0;">
                        <strong>❌ Error:</strong> Could not find data for file: {file_name}
                    </div>
                    """))
                continue
                
            file_data = uploaded_files_data[file_name]

            # Get file data from our stored mapping
            if file_name not in uploaded_files_data:
                with out4:
                    display(HTML(f"""
                    <div style="background: #f8d7da; border-radius: 8px; padding: 10px; margin: 5px 0;">
                        <strong>❌ Error:</strong> Could not find data for file: {file_name}
                    </div>
                    """))
                continue
            file_data = uploaded_files_data[file_name]
            
            # Get the file content from the ClientSideFile object
            if 'file_obj' in file_data:
                # Read the content from the ClientSideFile object
                file_content = file_data['file_content']
                time.sleep(0.5)
            else:
                with out4:
                    display(HTML(f"""
                    <div style="background: #f8d7da; border-radius: 8px; padding: 10px; margin: 5px 0;">
                        <strong>❌ Error:</strong> No data found in file: {file_name}
                    </div>
                    """))
                continue

            # Split filename for processing
            file_name_parts = file_name.split(".")
            file_type = file_name_parts[-1]
            file_name_old = "_".join(file_name_parts[:-1])
            measurement_type = "General"
            if sample_id in file_type_dict and file_name in file_type_dict[sample_id]:
                measurement_type = file_type_dict[sample_id][file_name]
            # Create new file name with appropriate measurement type
            new_file_name = f"{sample_id}-{file_name_old}.{measurement_type}.{file_type}"

            with out4:
                display(HTML(f"""
                <div style="background: #d1ecf1; border-radius: 8px; padding: 10px; margin: 5px 0; border-left: 4px solid #17a2b8;">
                    <strong>📁 Writing file:</strong> {new_file_name}
                </div>
                """))
                if not upload_folder:
                    display(HTML(f"""
                    <div style="background: #f8d7da; border-radius: 8px; padding: 15px; margin: 10px 0; border-left: 4px solid #dc3545;">
                        <h4 style="color: #721c24; margin: 0 0 5px 0;">🚫 Access Error</h4>
                        <p style="color: #721c24; margin: 0;">No write access to upload {upload_id}!</p>
                        <p style="color: #721c24; margin: 5px 0 0 0;">Please check access permissions and restart the notebook.</p>
                    </div>
                    """))
                    continue               
                else: 
                    print(f"📂 Upload folder: {upload_folder}")

            # Write the file
            target_directory = f"../{upload_folder}/{measurement_type}"
            if not os.path.exists(target_directory):
                os.makedirs(target_directory)
            with open(os.path.join(target_directory,new_file_name), "wb") as f:
                f.write(file_content)
    # Process uploads
    for upload_id in set(upload_ids):
        process_upload(upload_id, out4) # process uploads twice!
        time.sleep(1)
        process_upload(upload_id, out4)
        
    with out4:
        print("processed")
        out4.clear_output()
        display(HTML(f"""
        <div class="success-box">
            <h4 style="margin: 0 0 10px 0; color: white;">🎉 Upload Complete!</h4>
            <p style="margin: 0; color: white;">Successfully processed {len(set(upload_ids))} sample ID(s) with assigned files</p>
        </div>
        """))
    load_samples_in_upload()

        
        
def load_samples_in_upload():
    global out3, sample_id_buttons, output_areas, sample_files_dict
    out3.clear_output()

    # Clear previous buttons and output areas
    sample_id_buttons = []
    output_areas = {}  # Change to dictionary to track by sample_id

    # Get sample IDs for the selected batch
    upload_id = get_upload_id()
    entries = get_specific_entrytype_of_upload(url, token, upload_id, "EMIL_Sample")
    sample_ids = [s.get("lab_id") for s in entries if s.get("lab_id")] 
    sample_ids.sort()

    # Get sample IDs and names together to preserve mapping
    samples = [(s.get("lab_id"), s.get("name"))for s in entries if s.get("lab_id") and s.get("name")]

    # Sort based on lab_id
    samples.sort(key=lambda x: x[0])

    # Rebuild dict with correct mapping
    sample_files_dict = {sid: [] for sid, _ in samples}
 
    time.sleep(0.2)

    # Initialize the dictionary with empty arrays for each sample ID
    sample_files_dict = {sample_id: [] for sample_id in sample_ids}

    # Create a container for each sample ID with initially just the sample button
    for sample_id, sample_name in samples:
        sample_button = widgets.Button(  
            description=f"🧪 {sample_id} | {sample_name}",  
            layout=widgets.Layout(width='380px', height="45px"),
            style={'button_color': '#274b8e'},
            tooltip=f"Click to assign files to sample {sample_id}"
        )

        # Create output area to hold the SelectMultiple and Remove button
        output_area = widgets.Output(layout=widgets.Layout(height='auto', min_height='0px'))
        output_areas[sample_id] = output_area

        # Set up callback for the sample button
        sample_button.on_click(on_sample_button_first_click(sample_id, output_area))

        # Add to list
        sample_id_buttons.append(widgets.VBox([
            sample_button,
            output_area
        ], layout=widgets.Layout(
                    flex='0 0 auto',  # Don't shrink, don't grow, use natural size
            margin='8px 5px'    # Increased margin between buttons
        )))

    # Create the left panel (uploader section) with fixed layout
    left_panel = widgets.VBox([
        widgets.HTML('<div class="panel-header">📂 File Selection</div>'),
        widgets.VBox([
            file_count_display,
            file_input,  # Keep it but hidden
            widgets.HTML("<div style='margin: 15px 0; padding: 10px; background: #e3f2fd; border-radius: 8px;'><em>💡 Select files from the list below, then click on a sample ID to assign them</em></div>"),
            file_selector,
            unrecognized_files_widget
        ], layout=widgets.Layout(padding='15px'))
    ], layout=widgets.Layout(
        width='530px',
        height='800px',
        margin='5px'
    ), _dom_classes=['panel-container'])
    
    # Create the right panel (sample buttons) with scrollable layout
    right_panel = widgets.VBox([
        widgets.HTML('<div class="panel-header">🧪 Sample Assignment</div>'),
        dropdown_all_files,
        widgets.VBox(
            sample_id_buttons,
            layout=widgets.Layout(
                height='720px',
                overflow='scroll',  # Vertical scroll only
                padding='15px'
            )
        )
    ], layout=widgets.Layout(
        width='1000px',
        height='800px',
        margin='5px'
    ), _dom_classes=['panel-container'])

    # Display the sample ID buttons
    with out3:

        display(widgets.HBox([
            left_panel,
            right_panel
        ], layout=widgets.Layout(
            width='100%',
            align_items='flex-start'  # Align panels to the top
        )))

    # Show the upload_and_process button
    with upload_button_container:
        upload_button_container.clear_output()
        display(upload_and_process)
    
time.sleep(0.1)
load_samples_in_upload()
time.sleep(0.1)

try:
    file_input.file_info = [{'name': 'storm110.hdf', 'size': 3487, 'lastModified': 1684766593553, 'type': ''}]
    c = ClientSideFile(file_input,0,1)
    d = c.read()
except:
    file_input.file_info = []


# Register callbacks
upload_and_process.on_click(on_upload_file)
file_input.observe(on_file_input_change, names='file_info')  # Observe v_model changes
file_selector.observe(on_selection_change, names='value')

# Display the main interface
display(widgets.VBox([
    out4,
    out3,
    upload_button_container
]))

---

## 🕸️ Sample Network - Generate Graph representation of your Data


In [ ]:
from pyvis.network import Network
from IPython.display import HTML, clear_output
out5 = widgets.Output()
out6 = widgets.Output()

project = "Project title"

# Add styling for the graph section
display(HTML("""
<style>
    .graph-container {
        background: linear-gradient(135deg, #ff6b6b 0%, #ee5a52 100%);
        border-radius: 15px;
        padding: 20px;
        color: white;
        margin: 15px 0;
        box-shadow: 0 10px 30px rgba(255, 107, 107, 0.3);
        text-align: center;
    }
    
    .graph-button {
        background: linear-gradient(135deg, #4facfe 0%, #00f2fe 100%) !important;
        border: none !important;
        border-radius: 12px !important;
        padding: 15px 30px !important;
        color: white !important;
        font-weight: 600 !important;
        font-size: 16px !important;
        cursor: pointer !important;
        transition: all 0.3s ease !important;
        box-shadow: 0 6px 20px rgba(79, 172, 254, 0.4) !important;
    }
    
    .graph-button:hover {
        transform: translateY(-4px) !important;
        box-shadow: 0 12px 35px rgba(79, 172, 254, 0.6) !important;
    }
    
    .graph-output {
        background: white;
        border-radius: 15px;
        padding: 20px;
        margin: 20px 0;
        box-shadow: 0 8px 25px rgba(0, 0, 0, 0.1);
        border: 2px solid #e9ecef;
    }
</style>
"""))

def on_graph_clicked(b):
    with out6:
        clear_output()
    with out5:
        clear_output(wait=True)
        display(HTML("""
        <div style="background: #fff3cd; border: 1px solid #ffeaa7; border-radius: 10px; padding: 15px; margin: 10px 0; text-align: center;">
            <strong>🔄 Generating Graph...</strong><br>
            <small>Please wait while we analyze relationships and create the visualization</small>
        </div>
        """))

    upload_id = get_upload_id()
    project=upload_names.value
    samples = get_samples_in_upload(upload_id)
    sample_ids = [s["archive"]["data"]["lab_id"] for s in samples if "lab_id" in s["archive"]["data"]]
    
    # Enhanced network visualization
    net = Network(
        notebook=True,
        cdn_resources='in_line',
        directed=True,
        width="100%",
        height="600px",
        bgcolor="#ffffff",
        font_color="#333333"
    )
    
    sample_data = get_specific_data_of_sample(url, token, sample_ids)
    processes = {}
    
    for d in sample_data:
        if "samples" not in d["archive"]["data"] or "lab_id" not in d["archive"]["data"]["samples"][0]:
            continue
        label= d["archive"]["data"]["samples"][0].get("lab_id")
        if label not in processes:
            processes.update({label:{}})
        d["archive"]["data"].setdefault("method", "Measurement")
        processes[label].update({str(d["archive"]["data"]["method"])+" " +str(d["archive"]["data"]["name"]):make_link(d["archive"]["data"]["method"],
                                           d["archive"]["metadata"]["upload_id"], 
                                           d["archive"]["metadata"]["entry_id"])})
    for sid in samples:
        label = sid["archive"]["data"]["lab_id"]
        nick_name = sid["archive"]["data"]["name"]
        net.add_node(
            label,  
            label=nick_name, 
            title=make_link(label,upload_id, sid['entry_id']),
            group=0,
            color={'background': '#667eea', 'border': '#4c63d2'},
            font={'color': '#ffffff', 'size': 14}
        )
    for sid in samples:
        label = sid["archive"]["data"]["lab_id"]
        if "parent" in sid["archive"]["data"]:
            try:
                net.add_edge(label, sid["archive"]["data"]["parent"]["lab_id"], color={'color': '#274b8e'})
            except:
                with out6:
                    print(f"Error: The parent id {sid["archive"]["data"]["parent"]["lab_id"]} does not exist please check the parent id!")
                
        if not label in processes:
            continue
        for k,v in processes[label].items():
            net.add_node(
                k, 
                label=k, 
                title=v,
                group=1,
                size=15,
                color={'background': '#ff6b6b', 'border': '#ee5a52'},
                font={'color': '#ffffff', 'size': 12}
            )
            net.add_edge(k, label, color={'color': '#dc3545'})

    # Configure physics for better layout
    net.set_options("""
    {
        "physics": {
            "enabled": true,
            "stabilization": {"iterations": 100}
        },
        "nodes": {
            "borderWidth": 2,
            "borderWidthSelected": 4,
            "shapeProperties": {
                "useBorderWithImage": true
            }
        },
        "edges": {
            "width": 2,
            "smooth": {
                "type": "continuous"
            }
        }
    }
    """)

    net.show("overview.html", notebook=True)
    html = HTML("overview.html")
    upload_folder = get_upload_folder()
    
    with open(f"../{upload_folder}/overview.html", "w") as f:
        f.write(html.data)
    
    with out5:
        clear_output(wait=True)
        display(HTML(f"""
        <div class="graph-output">
            <h3 style="color: #333; margin: 0 0 15px 0; text-align: center;">{project}</h3>
            <div style="background: #e3f2fd; padding: 10px; border-radius: 8px; margin: 10px 0; text-align: center;">
                <small><strong>💡 Tip:</strong> Click and drag nodes to explore relationships. Hover and click on the link to see entry.</small>
            </div>
        </div>
        """))
        display(HTML(f"../{upload_folder}/overview.html"))
        display(HTML(f"""
        <div style="background: #d4edda; border: 1px solid #c3e6cb; border-radius: 10px; padding: 15px; margin: 15px 0; text-align: center;">
            <strong>✅ Graph Generated Successfully!</strong><br>
            <small>Graph saved to: {upload_folder}/overview.html</small>
        </div>
        """))


# Create styled button and container
button_graph = widgets.Button(
    description='Generate Graph',
    layout=widgets.Layout(width='200px', height='45px'),
    style={'button_color': '#dc3545'},
    tooltip='Create an interactive visualization of sample relationships'
)

button_graph.on_click(on_graph_clicked)

# Display with enhanced styling

display(widgets.VBox([
    button_graph, 
    widgets.HTML("""
    <div style="margin: 20px 0;">
        <div id="graph-output-container" class="graph-output" style="display: none;">
            <div id="graph-content"></div>
        </div>
    </div>
    """),
    out5
]))